In [ ]:
# ==============================
# PHASE 02 - CHUNK 01
# YOLOv8-Pose Setup & Sanity Check (FINAL)
# ==============================

# 1. INSTALL YOLOv8
!pip install -q ultralytics

# 2. IMPORTS
from ultralytics import YOLO
import cv2
import numpy as np
import os
import torch
from google.colab import drive

# 3. MOUNT DRIVE
drive.mount('/content/drive')

# 4. PATHS
PROJECT_ROOT = "/content/drive/MyDrive/Human-Fall-Recognition"
DATASET_ROOT = os.path.join(PROJECT_ROOT, "GUB-STFN-Fall-Dataset")
SKELETON_DIR = os.path.join(PROJECT_ROOT, "Skeleton_Raw")
os.makedirs(SKELETON_DIR, exist_ok=True)

# 5. GPU CHECK (MANDATORY)
print("\n--- Hardware Check ---")
if torch.cuda.is_available():
    print(f"✅ GPU Ready: {torch.cuda.get_device_name(0)}")
else:
    raise RuntimeError("❌ GPU not detected. Set Runtime → T4 GPU")

# 6. LOAD MODEL
print("\nLoading YOLOv8-Pose model...")
model = YOLO("yolov8n-pose.pt")
print("✅ Model loaded")

# 7. PICK ONE TEST VIDEO (SAFE)
test_class = "NoFall"
class_dir = os.path.join(DATASET_ROOT, test_class)
test_files = [f for f in os.listdir(class_dir) if f.endswith(".mp4")]

if not test_files:
    raise FileNotFoundError("❌ No .mp4 files found for sanity test")

test_video = os.path.join(class_dir, test_files[0])
print(f"\nTesting on video: {test_files[0]}")

# 8. SANITY EXTRACTION (LIMITED FRAMES)
cap = cv2.VideoCapture(test_video)
skeletons = []
count = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret or count >= 50:
        break

    results = model(frame, verbose=False)

    if results and results[0].keypoints is not None:
        kp = results[0].keypoints.xy.cpu().numpy()      # (P,17,2)
        conf = results[0].keypoints.conf.cpu().numpy()  # (P,17)

        if len(kp) > 0:
            skel = np.concatenate([kp[0], conf[0][:, None]], axis=1)
            skeletons.append(skel)

    count += 1

cap.release()
skeletons = np.array(skeletons)

print("-" * 40)
print(f"✅ Sanity skeleton shape: {skeletons.shape}")
print("Expected format: (N, 17, 3)")
print("STATUS: PASS — Ready for full batch extraction")

In [ ]:
# ==============================
# PHASE 02 - CHUNK 02
# Full Skeleton Extraction (Resume-Safe)
# ==============================

import os
import cv2
import numpy as np
import torch
from tqdm.notebook import tqdm
from ultralytics import YOLO
from google.colab import drive

# 1. MOUNT DRIVE
drive.mount('/content/drive')

# 2. PATHS
PROJECT_ROOT = "/content/drive/MyDrive/Human-Fall-Recognition"
DATASET_ROOT = os.path.join(PROJECT_ROOT, "GUB-STFN-Fall-Dataset")
OUTPUT_ROOT = os.path.join(PROJECT_ROOT, "Skeleton_Raw")
CLASS_NAMES = ["NoFall", "Faint", "Slip", "Trip"]

os.makedirs(OUTPUT_ROOT, exist_ok=True)

# 3. GPU CHECK
assert torch.cuda.is_available(), "GPU not available"
print("Using GPU:", torch.cuda.get_device_name(0))

# 4. LOAD MODEL
model = YOLO("yolov8n-pose.pt")

# 5. EXTRACT FUNCTION (TIME-PRESERVING)
def extract_skeletons(video_path):
    cap = cv2.VideoCapture(video_path)
    frames_data = []

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        results = model(frame, verbose=False)

        # Default zero skeleton → preserves timeline
        skeleton = np.zeros((17, 3))

        if results and results[0].keypoints is not None:
            kp = results[0].keypoints.xy.cpu().numpy()
            conf = results[0].keypoints.conf.cpu().numpy()
            if len(kp) > 0:
                skeleton = np.concatenate([kp[0], conf[0][:, None]], axis=1)

        frames_data.append(skeleton)

    cap.release()
    return np.array(frames_data)

# 6. MAIN LOOP (RESUME-SAFE)
for cls in CLASS_NAMES:
    cls_input = os.path.join(DATASET_ROOT, cls)
    cls_output = os.path.join(OUTPUT_ROOT, cls)
    os.makedirs(cls_output, exist_ok=True)

    videos = [v for v in sorted(os.listdir(cls_input)) if v.endswith(".mp4")]
    print(f"\nProcessing {cls}: {len(videos)} videos")

    for video in tqdm(videos, desc=cls):
        in_path = os.path.join(cls_input, video)
        out_path = os.path.join(cls_output, video.replace(".mp4", ".npy"))

        if os.path.exists(out_path):
            continue

        data = extract_skeletons(in_path)
        if len(data) > 0:
            np.save(out_path, data)

print("\nSkeleton extraction completed (or safely resumable)")